# 🎙️ Tamil Dialect Identification
## Using wav2vec 2.0 Embeddings + K-Means Clustering

**Authors:** Inbatamilan S (312323243073) | Leo Sanjay S (312423243098)  
**Institution:** St. Joseph's Institute of Technology, Chennai  
**Conference:** SpokenRoots@DravidianLangTech 2026 — ACL  

---
### 📌 How to Use This Notebook
1. Run **Step 1** to install dependencies
2. Run **Step 2** to load the wav2vec 2.0 model
3. **Option A** — Upload your own Tamil audio files and run the full pipeline
4. **Option B** — Use synthetic demo audio to test the pipeline instantly
5. Run **Step 5** to see clustering results and PCA visualization

> 💡 **Dialects:** Northern | Southern | Western | Central

---
## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install transformers torch librosa soundfile scikit-learn matplotlib seaborn -q
print("✅ All packages installed successfully!")

---
## Step 2: Import Libraries & Load wav2vec 2.0 Model

In [ ]:
import os
import torch
import librosa
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.preprocessing import normalize

print("📦 Libraries imported.")
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🖥️  Device: {'GPU ✅' if torch.cuda.is_available() else 'CPU (GPU recommended for speed)'}")

# ─── Load pretrained wav2vec 2.0 ───────────────────────────────────────────
MODEL_NAME = "facebook/wav2vec2-base"
print(f"\n⏳ Loading {MODEL_NAME} ... (first run may take ~1-2 min)")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model     = Wav2Vec2Model.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"✅ Model loaded on {device}!")

---
## Step 3: Core Functions — Preprocessing & Feature Extraction

In [ ]:
SAMPLE_RATE  = 16000   # wav2vec 2.0 requires 16kHz
MAX_DURATION = 5.0     # seconds — trim long audio
MAX_SAMPLES  = int(SAMPLE_RATE * MAX_DURATION)

DIALECT_NAMES = ["Northern", "Southern", "Western", "Central"]
COLORS        = ["#E63946", "#2A9D8F", "#E9C46A", "#457B9D"]

# ── 1. Audio Preprocessing ────────────────────────────────────────────────
def preprocess_audio(path):
    """
    Load → resample to 16 kHz mono → normalize amplitude → trim to MAX_DURATION.
    Returns a 1D numpy array.
    """
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    # Amplitude normalization
    max_val = np.abs(audio).max()
    if max_val > 0:
        audio = audio / max_val
    # Trim to fixed length
    if len(audio) > MAX_SAMPLES:
        audio = audio[:MAX_SAMPLES]
    return audio

# ── 2. wav2vec 2.0 Embedding Extraction ───────────────────────────────────
def extract_embedding(audio_array):
    """
    Pass preprocessed audio through wav2vec 2.0 encoder.
    Apply mean pooling over frame-level outputs → 768-dim utterance embedding.
    """
    inputs = processor(
        audio_array,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True
    )
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        outputs = model(input_values)

    # Mean pooling: (batch, time, 768) → (batch, 768)
    hidden_states = outputs.last_hidden_state          # shape: (1, T, 768)
    utterance_emb = hidden_states.mean(dim=1)          # shape: (1, 768)
    return utterance_emb.squeeze().cpu().numpy()       # shape: (768,)

# ── 3. Process a Single File ───────────────────────────────────────────────
def process_file(path):
    audio = preprocess_audio(path)
    emb   = extract_embedding(audio)
    return emb

print("✅ Core functions defined.")
print(f"   • Sample rate  : {SAMPLE_RATE} Hz")
print(f"   • Max duration : {MAX_DURATION}s  ({MAX_SAMPLES} samples)")
print(f"   • Embedding dim: 768  (wav2vec 2.0 base)")
print(f"   • Dialects     : {DIALECT_NAMES}")

---
## Step 4A: 📂 Upload Your Own Tamil Audio Files
> **Supported formats:** `.wav`, `.mp3`, `.flac`  
> Upload multiple files — one per utterance. Label them if you have ground-truth (optional).

In [ ]:
from google.colab import files
import io

print("📁 Upload your Tamil dialect audio files below:")
uploaded = files.upload()

uploaded_paths = []
for fname, data in uploaded.items():
    save_path = f"/content/{fname}"
    with open(save_path, 'wb') as f:
        f.write(data)
    uploaded_paths.append(save_path)
    print(f"  ✅ Saved: {fname}")

print(f"\n📊 Total files uploaded: {len(uploaded_paths)}")

In [ ]:
# ─── Extract embeddings from uploaded files ────────────────────────────────
embeddings_real  = []
filenames_real   = []
true_labels_real = []  # Fill in manually if you have ground truth

# ── OPTIONAL: manually set ground-truth labels here ──────────────────
# Map filename → dialect index (0=Northern, 1=Southern, 2=Western, 3=Central)
# Example: ground_truth_map = {"sample1.wav": 0, "sample2.wav": 1}
ground_truth_map = {}
# ─────────────────────────────────────────────────────────────────────

print("⏳ Extracting wav2vec 2.0 embeddings from uploaded files...\n")
for path in uploaded_paths:
    fname = os.path.basename(path)
    try:
        emb = process_file(path)
        embeddings_real.append(emb)
        filenames_real.append(fname)
        gt = ground_truth_map.get(fname, -1)
        true_labels_real.append(gt)
        print(f"  ✅ {fname:40s} → embedding shape: {emb.shape}")
    except Exception as e:
        print(f"  ❌ {fname}: {e}")

print(f"\n✅ Processed {len(embeddings_real)} files.")

---
## Step 4B: 🧪 OR — Run on Synthetic Demo Data (No Upload Needed)
> Generates synthetic audio signals to demonstrate the full pipeline instantly.  
> Simulates 4 dialect classes with different tonal characteristics.

In [ ]:
import soundfile as sf

def generate_synthetic_audio(dialect_idx, sample_idx, sr=16000, duration=3.0):
    """
    Simulate dialect-specific audio using different frequency patterns.
    Northern=low pitch, Southern=mid pitch, Western=high pitch, Central=mixed.
    Adds noise and harmonics to make embeddings dialect-discriminative.
    """
    t   = np.linspace(0, duration, int(sr * duration), endpoint=False)
    np.random.seed(dialect_idx * 100 + sample_idx)

    base_freqs = [110, 165, 220, 147]  # Hz per dialect
    f0   = base_freqs[dialect_idx] * (1 + 0.05 * np.random.randn())
    # Fundamental + harmonics
    sig  = (
        0.5  * np.sin(2 * np.pi * f0       * t) +
        0.25 * np.sin(2 * np.pi * f0 * 2   * t) +
        0.12 * np.sin(2 * np.pi * f0 * 3   * t) +
        0.08 * np.sin(2 * np.pi * f0 * 4   * t)
    )
    # Amplitude modulation (prosody simulation)
    amp_mod = 1 + 0.3 * np.sin(2 * np.pi * (2 + dialect_idx) * t)
    sig     = sig * amp_mod
    # Gaussian noise
    sig    += 0.05 * np.random.randn(len(t))
    # Normalize
    sig    /= np.abs(sig).max()
    return sig.astype(np.float32)

# ─── Generate and save synthetic audio ────────────────────────────────────
N_PER_CLASS    = 10   # samples per dialect
demo_dir       = "/content/demo_audio"
os.makedirs(demo_dir, exist_ok=True)

demo_paths, demo_labels = [], []

for d_idx, d_name in enumerate(DIALECT_NAMES):
    for s in range(N_PER_CLASS):
        audio = generate_synthetic_audio(d_idx, s)
        fname = f"{d_name.lower()}_{s:02d}.wav"
        fpath = os.path.join(demo_dir, fname)
        sf.write(fpath, audio, SAMPLE_RATE)
        demo_paths.append(fpath)
        demo_labels.append(d_idx)

print(f"✅ Generated {len(demo_paths)} synthetic audio files")
print(f"   ({N_PER_CLASS} per dialect × {len(DIALECT_NAMES)} dialects)")
print(f"   Saved to: {demo_dir}")

In [ ]:
# ─── Extract embeddings from demo audio ───────────────────────────────────
embeddings_demo  = []
filenames_demo   = []
true_labels_demo = []

print("⏳ Extracting wav2vec 2.0 embeddings from demo audio...\n")
for path, label in zip(demo_paths, demo_labels):
    fname = os.path.basename(path)
    try:
        emb = process_file(path)
        embeddings_demo.append(emb)
        filenames_demo.append(fname)
        true_labels_demo.append(label)
    except Exception as e:
        print(f"  ❌ {fname}: {e}")

print(f"\n✅ Extracted {len(embeddings_demo)} embeddings  (shape: 768-dim each)")

---
## Step 5: 🔵 K-Means Clustering — Dialect Assignment

In [ ]:
# ─── Select data source ───────────────────────────────────────────────────
# Change to embeddings_real / filenames_real / true_labels_real
# if you used Step 4A (uploaded your own files)
USE_DEMO = True  # ← Set False to use your uploaded files

if USE_DEMO:
    all_embeddings  = np.array(embeddings_demo)
    all_filenames   = filenames_demo
    all_true_labels = true_labels_demo
    print("📌 Using: DEMO synthetic audio")
else:
    all_embeddings  = np.array(embeddings_real)
    all_filenames   = filenames_real
    all_true_labels = true_labels_real
    print("📌 Using: Uploaded audio files")

print(f"   Embedding matrix shape: {all_embeddings.shape}")

# ─── L2 Normalize embeddings ──────────────────────────────────────────────
all_embeddings_norm = normalize(all_embeddings, norm='l2')

# ─── K-Means Clustering ───────────────────────────────────────────────────
K = 4  # Northern, Southern, Western, Central

kmeans = KMeans(
    n_clusters=K,
    init='k-means++',     # Smart initialization (Na et al., 2010)
    n_init=20,            # Run 20 times, pick best
    max_iter=500,
    random_state=42
)
cluster_ids = kmeans.fit_predict(all_embeddings_norm)

print(f"\n✅ K-Means clustering complete  (K={K})")
print(f"\n📊 Cluster distribution:")
for k in range(K):
    count = (cluster_ids == k).sum()
    print(f"   Cluster {k} → {count} utterances")

In [ ]:
# ─── Map clusters to dialect labels ───────────────────────────────────────
# Best-match assignment using Hungarian-style label mapping
from scipy.optimize import linear_sum_assignment

def best_cluster_to_label_map(cluster_ids, true_labels, n_clusters):
    """
    Find the cluster→dialect mapping that maximizes accuracy.
    Only used when true labels are available.
    """
    cost = np.zeros((n_clusters, n_clusters))
    for c in range(n_clusters):
        for l in range(n_clusters):
            cost[c, l] = -np.sum((cluster_ids == c) & (np.array(true_labels) == l))
    row, col = linear_sum_assignment(cost)
    return dict(zip(row, col))

# Check if true labels are available
has_labels = all_true_labels and (-1 not in all_true_labels)

if has_labels:
    cluster_map   = best_cluster_to_label_map(cluster_ids, all_true_labels, K)
    pred_labels   = [cluster_map[c] for c in cluster_ids]
    pred_dialects = [DIALECT_NAMES[l] for l in pred_labels]
    true_dialects = [DIALECT_NAMES[l] for l in all_true_labels]
    print("✅ Label mapping using ground-truth (Hungarian algorithm)")
    print(f"   Cluster→Dialect map: {cluster_map}")
else:
    # No ground truth — just assign dialects directly to cluster IDs
    cluster_map   = {i: i for i in range(K)}
    pred_labels   = [cluster_map[c] for c in cluster_ids]
    pred_dialects = [DIALECT_NAMES[l] for l in pred_labels]
    print("⚠️  No ground-truth labels — clusters assigned as-is")
    print("   (Results shown as cluster IDs; provide labels for F1 evaluation)")

---
## Step 6: 📈 Evaluation — Macro F1 Score & Classification Report

In [ ]:
if has_labels:
    macro_f1 = f1_score(all_true_labels, pred_labels, average='macro')
    random_baseline = 1.0 / K

    print("=" * 55)
    print(" EVALUATION RESULTS")
    print("=" * 55)
    print(f"  Macro F1 Score    : {macro_f1:.4f}")
    print(f"  Random Baseline   : {random_baseline:.4f}")
    print(f"  Improvement       : +{macro_f1 - random_baseline:.4f}")
    print("=" * 55)

    print("\n📋 Full Classification Report:\n")
    print(classification_report(
        all_true_labels,
        pred_labels,
        target_names=DIALECT_NAMES,
        digits=4
    ))

    # Confusion matrix
    cm = confusion_matrix(all_true_labels, pred_labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=DIALECT_NAMES, yticklabels=DIALECT_NAMES,
                linewidths=0.5)
    plt.title('Confusion Matrix — Tamil Dialect Identification', fontsize=13, fontweight='bold')
    plt.ylabel('True Dialect', fontsize=11)
    plt.xlabel('Predicted Dialect', fontsize=11)
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📁 Saved: /content/confusion_matrix.png")
else:
    print("⚠️  Skipping F1 evaluation — no ground-truth labels provided.")
    print("   Prediction counts per dialect:")
    for d, name in enumerate(DIALECT_NAMES):
        count = pred_dialects.count(name)
        print(f"   {name:10s}: {count} utterances")

---
## Step 7: 🔵 PCA Visualization — Embedding Cluster Space

In [ ]:
# ─── PCA: 768D → 2D ──────────────────────────────────────────────────────
pca       = PCA(n_components=2, random_state=42)
emb_2d    = pca.fit_transform(all_embeddings_norm)
var_ratio = pca.explained_variance_ratio_

print(f"PCA variance explained: PC1={var_ratio[0]*100:.1f}%  PC2={var_ratio[1]*100:.1f}%")

# ─── Plot: Predicted clusters ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2 if has_labels else 1, figsize=(14 if has_labels else 7, 5))
if not has_labels:
    axes = [axes]

# --- Left: Predicted ---
ax = axes[0]
for d_idx, (d_name, color) in enumerate(zip(DIALECT_NAMES, COLORS)):
    mask = np.array(pred_labels) == d_idx
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               c=color, label=d_name, alpha=0.75, s=80, edgecolors='white', linewidths=0.5)

# Plot cluster centroids
centroids_2d = pca.transform(normalize(kmeans.cluster_centers_, norm='l2'))
for k in range(K):
    mapped = cluster_map[k]
    ax.scatter(centroids_2d[k, 0], centroids_2d[k, 1],
               c=COLORS[mapped], s=250, marker='*',
               edgecolors='black', linewidths=1.2, zorder=5)

ax.set_title('HuBERT Embedding Clusters\n(Predicted Dialect Labels)', fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}%)', fontsize=10)
ax.set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}%)', fontsize=10)
patches = [mpatches.Patch(color=c, label=n) for c, n in zip(COLORS, DIALECT_NAMES)]
ax.legend(handles=patches, loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

# --- Right: True labels (if available) ---
if has_labels:
    ax2 = axes[1]
    for d_idx, (d_name, color) in enumerate(zip(DIALECT_NAMES, COLORS)):
        mask = np.array(all_true_labels) == d_idx
        ax2.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                    c=color, label=d_name, alpha=0.75, s=80, edgecolors='white', linewidths=0.5)
    ax2.set_title('HuBERT Embedding Clusters\n(Ground-Truth Dialect Labels)', fontsize=12, fontweight='bold')
    ax2.set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}%)', fontsize=10)
    ax2.set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}%)', fontsize=10)
    ax2.legend(handles=patches, loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)

plt.suptitle('Tamil Dialect Identification — wav2vec 2.0 + K-Means', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/content/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print("📁 Saved: /content/pca_clusters.png")

---
## Step 8: 📋 Sample Predictions Table

In [ ]:
import pandas as pd

results = []
for i, (fname, pred, cluster) in enumerate(zip(all_filenames, pred_dialects, cluster_ids)):
    row = {
        "File": fname,
        "Cluster": cluster,
        "Predicted Dialect": pred,
    }
    if has_labels:
        row["True Dialect"] = DIALECT_NAMES[all_true_labels[i]]
        row["Correct"]      = "✅" if pred == DIALECT_NAMES[all_true_labels[i]] else "❌"
    results.append(row)

df = pd.DataFrame(results)

# Show sample
print("📊 Sample Predictions (showing up to 20 rows):")
print(df.head(20).to_string(index=False))

# Save full CSV
df.to_csv('/content/dialect_predictions.csv', index=False)
print(f"\n📁 Full predictions saved: /content/dialect_predictions.csv")
print(f"   Total predictions: {len(df)}")

if has_labels:
    correct = (df['Correct'] == '✅').sum()
    print(f"   Correct predictions: {correct}/{len(df)} ({100*correct/len(df):.1f}%)")

---
## Step 9: 🎯 Single Audio Prediction — Test Any File

In [ ]:
def predict_dialect(audio_path, fitted_kmeans=kmeans, fitted_pca=pca, emb_matrix=all_embeddings_norm):
    """
    Predict the dialect of a single audio file using the fitted K-Means model.
    Returns predicted dialect name and confidence (distance to centroid).
    """
    print(f"\n🎙️  Analyzing: {os.path.basename(audio_path)}")

    # Extract embedding
    emb = process_file(audio_path)
    emb_norm = normalize(emb.reshape(1, -1), norm='l2')

    # Predict cluster
    cluster = fitted_kmeans.predict(emb_norm)[0]
    dialect_idx = cluster_map[cluster]
    dialect_name = DIALECT_NAMES[dialect_idx]

    # Distance to all centroids (lower = more confident)
    distances = np.linalg.norm(
        fitted_kmeans.cluster_centers_ - emb_norm, axis=1
    )
    # Convert to confidence scores (softmax-like)
    inv_dist = 1.0 / (distances + 1e-6)
    confidence = inv_dist / inv_dist.sum()

    print(f"\n  ┌─────────────────────────────────────┐")
    print(f"  │  Predicted Dialect: {dialect_name:12s}    │")
    print(f"  │  Cluster ID       : {cluster}                    │")
    print(f"  └─────────────────────────────────────┘")
    print(f"\n  Confidence scores:")
    for d, (name, conf) in enumerate(zip(DIALECT_NAMES, [confidence[cluster_map_inv.get(d, d)] for d in range(K)])):
        bar = '█' * int(conf * 30)
        marker = " ←" if name == dialect_name else ""
        print(f"    {name:10s}: {bar:30s} {conf*100:.1f}%{marker}")

    return dialect_name, confidence

# Build inverse cluster map for display
cluster_map_inv = {v: k for k, v in cluster_map.items()}

# ─── Test on first demo file ──────────────────────────────────────────────
if demo_paths:
    test_file = demo_paths[5]  # Pick a demo file
    predict_dialect(test_file)

---
## Step 10: 🎵 Upload & Predict a Single Audio File

In [ ]:
print("📁 Upload a single Tamil audio file to predict its dialect:")
new_upload = files.upload()

for fname, data in new_upload.items():
    path = f"/content/test_{fname}"
    with open(path, 'wb') as f:
        f.write(data)
    dialect, conf = predict_dialect(path)

---
## Step 11: 📥 Download All Results

In [ ]:
from google.colab import files as colab_files

# Download predictions CSV
print("📥 Downloading results...")
colab_files.download('/content/dialect_predictions.csv')
colab_files.download('/content/pca_clusters.png')
if os.path.exists('/content/confusion_matrix.png'):
    colab_files.download('/content/confusion_matrix.png')
print("✅ Done!")

---
## 📌 Summary

| Component | Details |
|-----------|--------|
| **Model** | facebook/wav2vec2-base (self-supervised) |
| **Embedding** | 768-dim, mean pooled over frame-level outputs |
| **Clustering** | K-Means (K=4, k-means++ init, 20 runs) |
| **Dialects** | Northern · Southern · Western · Central |
| **Evaluation** | Macro F1 Score (baseline = 0.25 for 4 classes) |
| **Paper result** | Macro F1 = 0.27 on DravidianLangTech 2026 test set |
| **Pipeline** | Fully unsupervised — no labeled data required |

### Future Improvements
- 🔧 Fine-tune wav2vec 2.0 on Tamil dialect speech (supervised)
- 🎵 Add prosodic features (pitch, rhythm, duration)
- 🤝 Semi-supervised learning with a small labeled set
- 🧠 Try HuBERT or XLS-R for richer cross-lingual embeddings

---
*Inbatamilan S & Leo Sanjay S — St. Joseph's Institute of Technology, Chennai — 2026*